# Build a Minimal Sierra — a guardrailed customer support agent

[Sierra](https://sierra.ai) builds production customer-service agents for
brands: agents that actually resolve issues — looking up orders, processing
refunds, making account changes — against real business systems, under strict
policy, with a clean handoff to humans when they hit their limits.

What makes systems like this trustworthy isn't the model. It's the
**architecture around the model**:

```
customer ──▶ ┌──────────────────────────┐        ┌────────────────────┐
             │  support agent           │ ─────▶ │ business systems   │
             │  policy in the prompt    │ tools  │ (orders, refunds)  │
             └──────┬───────────────────┘        │ policy in the CODE │
                    │ escalate_to_human          └─────────┬──────────┘
                    ▼                                      ▼
             structured ticket                       audit log of
             for a human agent                       every action
```

Three load-bearing ideas we'll replicate:

1. **Grounded actions** — the agent can only know and do what its tools allow.
   It cannot invent an order or a refund; it can only call `get_order` and
   `process_refund`.
2. **Two-layer guardrails** — policy lives in the system prompt (how to
   behave) *and* in the tool code (hard limits that hold even if the model is
   talked into anything).
3. **Escalation as a first-class action** — failure mode is a structured
   ticket to a human, never improvisation.

## Setup

In [ ]:
%pip install -q vidbyte-sdk python-dotenv requests

import os
from dotenv import load_dotenv

load_dotenv()             # .env next to this notebook
load_dotenv("../../.env") # or at the repo root

PROVIDER = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai")
MODEL = os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), \
    "Set OPENAI_API_KEY (or ANTHROPIC_API_KEY + provider overrides) in .env first."

## Step 1 — The business systems

A toy order database, plus the two records every production deployment keeps:
an **audit log** (every tool action, recorded in code, not by the model) and
an **escalation queue**.

In [ ]:
from datetime import datetime, timezone

ORDERS = {
    "A-1001": {"email": "dana@example.com", "item": "Trail Running Shoes", "amount": 89.00, "status": "delivered", "refunded": False},
    "A-1002": {"email": "dana@example.com", "item": "Carbon Trekking Poles", "amount": 240.00, "status": "delivered", "refunded": False},
    "A-1003": {"email": "sam@example.com", "item": "Headlamp", "amount": 35.00, "status": "shipped", "refunded": False},
}

AUDIT_LOG: list[dict] = []
ESCALATIONS: list[dict] = []

REFUND_LIMIT = 100.00  # hard policy: above this, a human must approve


def audit(action: str, **details):
    AUDIT_LOG.append({"ts": datetime.now(timezone.utc).isoformat(), "action": action, **details})

## Step 2 — The tools, with policy in the code

Look closely at `process_refund`: the \$100 limit and the double-refund check
are enforced **in Python**, not in the prompt. The model can be persuaded;
the `if` statement cannot. The tool refuses and tells the agent *why*, which
steers it toward the correct next action — escalation.

In [ ]:
from vidbyte import tool


@tool
def find_orders(email: str) -> list[dict]:
    """Find all orders for a customer email. Returns order_id, item, amount,
    status, and refund state for each."""
    audit("find_orders", email=email)
    return [
        {"order_id": oid, **{k: v for k, v in o.items() if k != "email"}}
        for oid, o in ORDERS.items() if o["email"].lower() == email.lower()
    ]


@tool
def get_order(order_id: str) -> dict:
    """Fetch one order by its order_id."""
    audit("get_order", order_id=order_id)
    order = ORDERS.get(order_id)
    return {"order_id": order_id, **order} if order else {"error": f"no order {order_id}"}


@tool
def process_refund(order_id: str, reason: str) -> dict:
    """Refund an order in full. Only succeeds within policy: order must exist,
    must not already be refunded, and must be at or under the auto-refund limit."""
    order = ORDERS.get(order_id)
    if order is None:
        audit("refund_rejected", order_id=order_id, why="unknown order")
        return {"ok": False, "why": f"no order {order_id}"}
    if order["refunded"]:
        audit("refund_rejected", order_id=order_id, why="already refunded")
        return {"ok": False, "why": "this order was already refunded"}
    if order["amount"] > REFUND_LIMIT:
        audit("refund_rejected", order_id=order_id, why="over auto-refund limit")
        return {"ok": False, "why": f"amount {order['amount']} exceeds the {REFUND_LIMIT} auto-refund limit — escalate to a human"}
    order["refunded"] = True
    audit("refund_processed", order_id=order_id, amount=order["amount"], reason=reason)
    return {"ok": True, "refunded": order["amount"]}


@tool
def escalate_to_human(customer_email: str, summary: str, severity: str) -> dict:
    """Open a ticket for a human agent. severity is 'low', 'normal', or 'high'.
    Use whenever a request is outside policy or you cannot resolve it with your tools."""
    ticket = {
        "ticket_id": f"T-{len(ESCALATIONS) + 1:04d}",
        "customer_email": customer_email,
        "summary": summary,
        "severity": severity if severity in ("low", "normal", "high") else "normal",
    }
    ESCALATIONS.append(ticket)
    audit("escalated", **ticket)
    return ticket

## Step 3 — The agent, with policy in the prompt

The prompt layer handles everything the code layer can't: identity
verification, tone, scope discipline, and *when* to reach for escalation.

In [ ]:
from vidbyte import BaseAgent

SUPPORT_PROMPT = """You are a customer support agent for Summit Outfitters.

Policy:
- Verify identity first: ask for the customer's email and look up their orders
  before discussing any order details. Never reveal another customer's data.
- Only state facts that came from your tools. If a tool doesn't know, you don't know.
- Refunds happen only through process_refund. If it refuses — over the limit,
  already refunded, anything — do NOT argue, retry, or promise outcomes:
  escalate_to_human with a precise summary, and tell the customer a human will
  follow up within one business day.
- Stay in scope: orders, refunds, and order status. Anything else gets escalated.
- Tone: warm, concise, no corporate filler. One question at a time."""

support = BaseAgent(
    name="support-agent",
    system_prompt=SUPPORT_PROMPT,
    provider=PROVIDER,
    model_name=MODEL,
    tools=[find_orders, get_order, process_refund, escalate_to_human],
    max_tool_rounds=6,
)

## Step 4 — Happy path: an in-policy refund

Dana wants a refund on her \$89 shoes. That's under the limit, so the agent
should verify identity, find the order, and resolve it end-to-end with no
human involved — the core Sierra value proposition.

In [ ]:
reply = support.run("Hi, I'd like a refund on my running shoes. My email is dana@example.com.")
print(reply.content)

## Step 5 — Guardrail path: an out-of-policy refund

Same customer, but the \$240 trekking poles are over the auto-refund limit.
Watch the layers work: the model *tries* the refund, **the tool refuses in
code**, and the prompt policy routes the failure into a structured escalation
instead of an argument or a hallucinated promise.

In [ ]:
reply = support.run("Actually, can you also refund my trekking poles? Same email.")
print(reply.content)

## Step 6 — The paper trail

The audit log and escalation queue are what make an agent like this
deployable: every action the agent took is recorded by the code layer,
in order, regardless of what the conversation claimed.

In [ ]:
import json

print("AUDIT LOG")
for entry in AUDIT_LOG:
    print(" ", json.dumps(entry))

print("\nESCALATION QUEUE")
for ticket in ESCALATIONS:
    print(" ", json.dumps(ticket))

print("\nORDER STATE")
for oid, order in ORDERS.items():
    print(f"  {oid}: refunded={order['refunded']}")

## What production Sierra adds

- **An agent OS** — versioned policies, A/B-tested prompts, regression evals on
  past conversations before any change ships.
- **Real system integrations** — OMS, CRM, auth — behind the same tool-shaped
  contracts you built here.
- **Supervision** — sampled conversation review, automatic QA scoring,
  real-time handoff to live human chat (not just a ticket queue).
- **Multi-channel** — the same agent core behind chat, email, and voice.

Things to try: add an `update_shipping_address` tool with its own code-level
guardrail (only before shipment); make a hostile customer cell that tries to
talk the agent over the refund limit and watch the code layer hold; route
`severity` from a classifier instead of the model's judgment.